# ETL Silver - Nivel diario ANA

Genera una fila diaria por `fecha + codigoestacao` para la estacion target `74100000`.

In [ ]:
from datetime import timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F

BRONZE_TABLE = 'weather.bronze.nivel_ana'
TARGET_TABLE = 'weather.silver.river_levels_daily'
QUALITY_TABLE = 'weather.silver.attribute_quality'
TARGET_STATION = '74100000'
THRESHOLD_PCT = 0.90

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('incremental_lookback_days', '14')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
except Exception:
    load_mode = 'incremental'
    incremental_lookback_days = 14

print(f'load_mode={load_mode}, incremental_lookback_days={incremental_lookback_days}')

In [ ]:
def parse_decimal(column_name):
    return F.regexp_replace(F.trim(F.col(column_name).cast('string')), ',', '.').cast('double')


def apply_incremental_window(df):
    if load_mode == 'full':
        return df

    max_target_fecha = spark.table(TARGET_TABLE).agg(F.max('fecha').alias('max_fecha')).first()['max_fecha']
    if max_target_fecha is None:
        return df

    start_date = max_target_fecha - timedelta(days=incremental_lookback_days)
    print(f'Processing Silver level from {start_date}')
    return df.filter(F.col('fecha') >= F.lit(start_date))


def build_quality(df, attribute_name, notes):
    return (
        df.agg(
            F.min('fecha').alias('evaluation_start_date'),
            F.max('fecha').alias('evaluation_end_date'),
            F.countDistinct(F.when(F.col(attribute_name).isNotNull(), F.col('fecha'))).cast('bigint').alias('observed_days'),
        )
        .withColumn('expected_days', F.when(F.col('evaluation_start_date').isNull(), F.lit(0)).otherwise(F.datediff(F.col('evaluation_end_date'), F.col('evaluation_start_date')) + F.lit(1)).cast('bigint'))
        .withColumn('missing_days', F.greatest(F.col('expected_days') - F.col('observed_days'), F.lit(0)).cast('bigint'))
        .withColumn('missing_pct', F.when(F.col('expected_days') == 0, F.lit(1.0)).otherwise(F.col('missing_days') / F.col('expected_days')))
        .withColumn('threshold_pct', F.lit(THRESHOLD_PCT))
        .withColumn('is_usable', F.col('missing_pct') <= F.col('threshold_pct'))
        .withColumn('source_layer', F.lit('silver'))
        .withColumn('source_table', F.lit(TARGET_TABLE))
        .withColumn('source_name', F.lit('river_levels_daily'))
        .withColumn('attribute_name', F.lit(attribute_name))
        .withColumn('grain', F.lit('global_source_daily'))
        .withColumn('evaluated_at', F.current_timestamp())
        .withColumn('notes', F.lit(notes))
        .withColumn('created_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
        .select('source_layer', 'source_table', 'source_name', 'attribute_name', 'grain', 'evaluation_start_date', 'evaluation_end_date', 'expected_days', 'observed_days', 'missing_days', 'missing_pct', 'threshold_pct', 'is_usable', 'evaluated_at', 'notes', 'created_at', 'updated_at')
    )


def merge_quality(quality_df):
    DeltaTable.forName(spark, QUALITY_TABLE).alias('t').merge(
        quality_df.alias('s'),
        't.source_table = s.source_table AND t.attribute_name = s.attribute_name AND t.grain = s.grain',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


def merge_daily(daily_df):
    if daily_df.limit(1).count() == 0:
        print('No level rows to merge')
        return

    DeltaTable.forName(spark, TARGET_TABLE).alias('t').merge(
        daily_df.alias('s'),
        't.fecha = s.fecha AND t.codigoestacao = s.codigoestacao',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [ ]:
bronze = (
    spark.table(BRONZE_TABLE)
    .select('codigoestacao', 'Data_Hora_Medicao', 'Cota_Adotada')
    .withColumn('codigoestacao', F.col('codigoestacao').cast('string'))
    .withColumn('medicao_ts', F.to_timestamp('Data_Hora_Medicao'))
    .withColumn('fecha', F.to_date('medicao_ts'))
    .withColumn('nivel_cm', parse_decimal('Cota_Adotada'))
    .filter(F.col('codigoestacao') == F.lit(TARGET_STATION))
    .filter(F.col('fecha').isNotNull())
)

bronze = apply_incremental_window(bronze)

daily = (
    bronze.groupBy('fecha', 'codigoestacao')
    .agg(
        F.avg('nivel_cm').alias('nivel_media_cm'),
        F.count('*').cast('bigint').alias('registros_total'),
        F.count('nivel_cm').cast('bigint').alias('registros_validos'),
        F.min('medicao_ts').alias('first_medicao_ts'),
        F.max('medicao_ts').alias('last_medicao_ts'),
    )
    .withColumn('nivel_media_m', F.col('nivel_media_cm') / F.lit(100.0))
    .withColumn('source_table', F.lit(BRONZE_TABLE))
    .withColumn('processed_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .select('fecha', 'codigoestacao', 'nivel_media_cm', 'nivel_media_m', 'registros_total', 'registros_validos', 'first_medicao_ts', 'last_medicao_ts', 'source_table', 'processed_at', 'updated_at')
)

merge_daily(daily)

full_daily_for_quality = spark.table(TARGET_TABLE).filter(F.col('codigoestacao') == F.lit(TARGET_STATION))
merge_quality(build_quality(full_daily_for_quality, 'nivel_media_cm', 'Nivel diario medio para codigoestacao 74100000'))

spark.table(TARGET_TABLE).filter(F.col('codigoestacao') == F.lit(TARGET_STATION)).agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows')).show()